# Discrimination pipeline

Run the built-in **discrimination** paradigm. Real StimPy sessions are resolved via `config.paths`; **if none are found (or a session fails to parse) we print the error and fall back to a simulated session** so every analysis cell still runs.

**For real data:** edit `~/.piepy/config.json` so `paths.presentation` / `paths.analysis` point at your dirs. (Real sessions need their full artifacts — e.g. opto-pattern images for opto sessions — or parsing will raise.)

In [ ]:
import os
import polars as pl

from piepy.core.config import config
from piepy.core.registry import get_session_class
from piepy.core.hub import Hub
from piepy.tasks.wheel_discrimination.wheelDiscriminationSession import WheelDiscriminationSession

from piepy.viz.plots import psychometric
from piepy.viz.trial.wheel_detection import trial_snapshot

pres = config.paths["presentation"][0]
print('presentation dir:', pres)

## Parse a single session
Builds the Session, parses each run, and stacks them onto one session clock. Wrapped defensively so a real-data hiccup prints a clear error instead of aborting.

In [ ]:
sess = WheelDiscriminationSession("260420_RO108_opto120_1stim_AMPM__no_cam_VO")
df = sess.analyze(load_flag=True)

## Plotting from a single Run/Session

### Plotting from a dataframe

In [ ]:
pr = psychometric(data=df,
                  x="diff_width",
                  outcome="right_choice",
                  compare="opto",
                  success=1,
                  fit_curve=True,
                  model="logistic",
                  palette=("#DD7703","#0164E5"),
                  label='Opto ',
                  preset="psychometric_discrimination")

#### You can pass ```kwargs``` that override the visual properties of the plots

This requires a bit of knowledge of the composition of the plot. Psychometric plot is made from an ```errorbar``` and ```line``` plot. You can target the style of these components by prefixing your ```kwargs``` with their names, e.g. ```errorbar_linewidth```, ```line_linestyle```.

In [ ]:
pr = psychometric(data=df,
                  x="diff_width",
                  outcome="outcome",
                  compare="opto",
                  success="correct",
                  fit_curve=True,
                  model="logistic",
                  palette=("#DD7703","#0164E5"),
                  errorbar_linewidth=9,
                  line_linestyle=":",
                  label='Opto ')

#### The returned ```PlotResult``` object has the data, statistical tests(if applicable) and the (fig,ax) tuple

In [ ]:
pr

### Plotting with accessor

You can pass a ```filterer``` argument to filter the data

In [ ]:
pr = sess.viz.reaction_time_cloud(filterer={"outcome":"correct"},
                                  x="diff_width",
                                  value="response_time",
                                  compare="opto",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=50,
                                  label='Opto',
                                  dodge_width=0.2,
                                  violin_widths=0.2,
                                  violin_showextrema=False,
                                  violin_showmedians=True,
                                  width=0.05,
                                  scatter_s=50,
                                  scatter_linewidth=0.3,
                                  scatter_edgecolor="#FFFFFF")

In [ ]:
pr = sess.viz.reaction_time_dist(filterer={"outcome":"correct",
                                           "diff_width":32},
                                  value="response_time",
                                  comparing="opto",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=5,
                                  label='Opto ',
                                  alpha=0.8)

## Cohort across many sessions (`Hub`)
`Hub` runs each session in parallel and stacks them into one cohort table. It already isolates per-session failures (a bad session warns and is skipped).

In [ ]:
import os
for r in os.listdir("/Users/kaan/data/virginie_opto_patterns"):
    if "RO1" in r:
        print(r)

In [ ]:
sessions = ["250213_VB101_discrim_opto120_AMPM__no_cam_VO",
            "250325_VB101_discrim_opto120_ALRL__no_cam_VO",
            "250213_VB101_discrim_opto120_AMPM__no_cam_VO",
            "250215_VB101_discrim_opto120_LILM__no_cam_VO",
            "250404_VB101_discrim_opto120_V1__no_cam_VO",
            "250401_VB101_discrim_opto120_AMPM__no_cam_VO",
            "250217_VB101_discrim_opto120_V1__no_cam_VO"
            ]

In [ ]:
sessions_RO = [
"260420_RO108_opto120_1stim_AMPM__no_cam_VO",
"260422_RO108_opto120_1stim_LMLI__no_cam_VO",
"260417_RO108_opto120_1stim_HVA__no_cam_VO",
"260226_RO109_opto120_1stim_Somato__no_cam_VO",
"260303_RO109_opto120_1stim_LMLI__no_cam_VO",
# "260225_RO109_opto120_1stim_V1__no_cam_VO", # has 2 opto_pattern values, was it a double target experiment?
"260305_RO109_opto120_1stim_ALRL__no_cam_VO",
"260423_RO108_opto120_1stim_V1__no_cam_VO",
"260421_RO108_opto120_1stim_ALRL__no_cam_VO",
"260306_RO109_opto120_1stim_AMPM__no_cam_VO",
"260227_RO109_opto120_1stim_HVA__no_cam_VO"]

In [ ]:
cohort = None
hub = Hub("wheel_discrimination")
hub.initialize([os.path.basename(s) for s in sessions_RO], load_sessions=True)
cohort = hub.data
print('cohort:', None if cohort is None else cohort.shape)

In [ ]:
cohort_HVA = cohort.filter(pl.col("area")=="HVA")


In [ ]:
pr = hub.viz.psychometric(filterer={"area":"HVA"},
                          x="diff_width",
                          outcome="right_choice",
                          compare="opto",
                          success=1,
                          fit_curve=True,
                          average_over=None,
                          model="logistic",
                          palette=("#090909","#0164E5"),
                          label='Opto ',
                          preset="psychometric_discrimination")

In [ ]:
pr = hub.viz.reaction_time_cloud(filterer={"outcome":"correct",
                                           "area":"HVA"},
                                  x="diff_width",
                                  value="response_time",
                                  compare="opto",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=50,
                                  average_over=None,
                                  label='Opto',
                                  dodge_width=5,
                                  violin_widths=5,
                                  violin_showextrema=False,
                                  violin_showmedians=True,
                                  width=5,
                                  scatter_s=50,
                                  scatter_linewidth=0.3,
                                  scatter_edgecolor="#FFFFFF",
                                  preset="reaction_time_cloud_discrimination")

In [ ]:
pr = hub.viz.reaction_time_dist(filterer={"outcome":"correct",
                                           "diff_width":-10},
                                  value="response_time",
                                  comparing="opto",
                                  palette=("#090909","#4C4CC7"),
                                  bin_width=5,
                                  label='Opto ',
                                  alpha=0.8)